# MyFarmWise Consultant: Plant Pathology & Attention-CNN Experiments
## Milestone Week 6 Progress Demonstration — 29 October 2026

This research notebook documents the computer vision subsystem of **MyFarmWise Consultant**:
1. **38-Class Pathology Catalog:** Mapping 38 diseases across 14 crops.
2. **Crop-Conditioned Filtering:** Restricting disease search spaces using crop context (Table 4.2 Contribution 1).
3. **Field-Realistic Augmentations:** Simulating motion blur, shadows, and outdoor clutter.
4. **Attention-CNN Architecture:** Squeeze-and-Excitation (SE) identity blocks (Figure 4.1).
5. **Grad-CAM Integration:** Mathematical formulation and gradient hooks on `last_se_conv`.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Set project directory
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from config.settings import CNN_INPUT_SHAPE, NUM_DISEASE_CLASSES, SAMPLE_IMAGES_DIR
from src.disease.preprocessing import load_and_preprocess_image, apply_field_realistic_augmentation
from src.disease.model import build_se_attention_cnn, get_model_architecture_summary
from src.disease.predict import load_disease_catalog, get_crop_conditioned_classes
from src.disease.gradcam import generate_gradcam_explanation
print('All disease diagnostic modules loaded successfully!')

## 1. Disease Catalog & Crop-Conditioned Filtering (Table 4.2)
Existing disease classifiers test blindly across all 38 classes, leading to out-of-domain errors (e.g. diagnosing Apple Scab on a Tomato leaf). MyFarmWise resolves this by filtering candidate classes to the selected crop.

In [ ]:
catalog = load_disease_catalog()
print(f'Total Registered Disease Classes: {len(catalog)}')

# Test crop conditioning
for target_crop in ['Tomato', 'Potato', 'Apple', 'Corn']:
    matched = get_crop_conditioned_classes(target_crop)
    print(f'Crop: {target_crop:<10} | Candidate Classes: {len(matched):<2} | Sample: {catalog[matched[0]]["display_name"]}')

## 2. Field-Realistic Augmentation Benchmarking (Table 4.2 Contribution 3)
To bridge the lab-to-field domain shift (where accuracy drops from ~99% to 17-46% on outdoor imagery), we implement four field transformations.

In [ ]:
sample_files = list(SAMPLE_IMAGES_DIR.glob('*.JPG'))
if sample_files:
    test_img = Image.open(sample_files[0])
    aug_results = apply_field_realistic_augmentation(test_img)
    
    fig, axes = plt.subplots(1, 5, figsize=(16, 4))
    axes[0].imshow(test_img)
    axes[0].set_title('Original Leaf')
    axes[1].imshow(aug_results['motion_blur'])
    axes[1].set_title('Motion Blur (Hand Shake)')
    axes[2].imshow(aug_results['shadow_casting'])
    axes[2].set_title('Canopy Shadow')
    axes[3].imshow(aug_results['illumination_shift'])
    axes[3].set_title('Sunlight Shift')
    axes[4].imshow(aug_results['background_clutter'])
    axes[4].set_title('Background Clutter')
    for ax in axes: ax.axis('off')
    plt.tight_layout()
    plt.show()

## 3. SE-Attention CNN Model Architecture (Figure 4.1)
Instantiating the Attention-CNN with 3 Squeeze-and-Excitation identity blocks for channel recalibration.

In [ ]:
summary = get_model_architecture_summary()
print(f'Model Architecture: {summary["model_name"]}')
print(f'Input Dimensions:   {summary["input_shape"]}')
print(f'Total Parameters:   {summary["total_parameters"]:,}')
print(f'Trainable Params:   {summary["trainable_parameters"]:,}')
print(f'Grad-CAM Hook:      {summary["gradcam_target_layer"]}')

for s in summary['stages']:
    print(f' - {s["stage"]:12}: {s["output_dim"]:16} -> {s["details"]}')

## 4. Grad-CAM Explainable AI Hook Verification
Mathematical formulation:
$$\alpha_k^c = \frac{1}{Z} \sum_i \sum_j \frac{\partial y^c}{\partial A_{ij}^k}$$
$$L_{\text{Grad-CAM}}^c = \text{ReLU}\left(\sum_k \alpha_k^c A^k\right)$$
The hook targets the `last_se_conv` layer of the third SE identity block.

In [ ]:
if sample_files:
    grad_demo = generate_gradcam_explanation(sample_files[0])
    print('Grad-CAM Engine Status:', grad_demo['status'])
    if 'mathematical_foundation' in grad_demo:
        for k, v in grad_demo['mathematical_foundation'].items():
            print(f'  {k}: {v}')